<a href="https://colab.research.google.com/github/sersayser/kopyala-yapistir/blob/main/Ciftci_Qwen3_6_8B_full_BF16_fine_tune_ipynb_adl%C4%B1_not_defterinin_kopyas%C4%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, json, torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
)

# === Config ===
MODEL_NAME = 'Qwen/Qwen3-8B'  # 27B Colab'da OOM. 8B full fine-tune sığar (16 GB model)
QA_FILE = '/content/drive/MyDrive/ciftci/qa_chatml.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/ciftci/qwen3-sft-bf16'

assert os.path.exists(QA_FILE), f'{QA_FILE} yok'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import os

directory = '/content/drive/MyDrive/ciftci/'
if os.path.exists(directory):
    print(f"{directory} dizinindeki dosyalar:")
    files = os.listdir(directory)
    for f in files:
        print(f"- {f}")
else:
    print(f"{directory} dizini bulunamadı. Google Drive'ın bağlı olduğundan emin olun.")

/content/drive/MyDrive/ciftci/ dizinindeki dosyalar:
- qa_chatml.jsonl
- qwen3_finetune_colab.ipynb
- qwen3_full_bf16_colab.ipynb
- qwen3-sft-bf16
- final
- nvme_offload
- ds_config_zero3.json


In [ ]:
# === 1. Tokenizer ===
print('Tokenizer yükleniyor...')
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

Tokenizer yükleniyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [ ]:
# === 2. Model — BF16, no quant, no LoRA, GPU'ya yükle ===
print(f'Model yükleniyor: {MODEL_NAME}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    attn_implementation='sdpa',
    device_map='auto',
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
n_params = sum(p.numel() for p in model.parameters())
print(f'Params: {n_params/1e9:.1f}B (TAMAMI eğitilecek, LoRA YOK)')

Model yükleniyor: Qwen/Qwen3-8B


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Params: 8.2B (TAMAMI eğitilecek, LoRA YOK)


In [ ]:
# === 3. Veri yükle + tokenize ===
print('Veri yükleniyor...')
records = []
with open(QA_FILE) as f:
    for line in f:
        line = line.strip()
        if line:
            r = json.loads(line)
            if 'messages' in r:
                records.append(r)
print(f'{len(records):,} kayıt')

def tokenize(ex):
    text = tok.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    enc = tok(text, max_length=2048, truncation=True, padding=False)
    enc['labels'] = enc['input_ids'].copy()
    return enc

ds = Dataset.from_list(records)
ds = ds.map(tokenize, remove_columns=['messages'], num_proc=2)
split = ds.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"]):,} | Val: {len(split["test"]):,}')

# === 4. Trainer config ===
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,
    learning_rate=2e-5,
    num_train_epochs=3,
    warmup_steps=50,
    weight_decay=0.01,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='adafactor',
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    eval_strategy='steps',
    eval_steps=200,
    seed=42,
    report_to='none',
    dataloader_num_workers=2,
)

collator = DataCollatorForLanguageModeling(tok, mlm=False)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=collator,
)

Veri yükleniyor...
19,336 kayıt


Map (num_proc=2):   0%|          | 0/19336 [00:00<?, ? examples/s]

Train: 18,369 | Val: 967


In [ ]:
# === 5. Resume varsa kaldığı yerden ===
from pathlib import Path
ckpts = sorted(Path(OUTPUT_DIR).glob('checkpoint-*'),
               key=lambda p: int(p.name.split('-')[1]), reverse=True)
resume = str(ckpts[0]) if ckpts else None
print(f'\nResume: {resume or "sıfırdan"}')

# === 6. EĞİT ===
print('\n' + '='*70)
print('Training başladı — loss değerleri akmalı (~10 sn/step)')
print('='*70 + '\n')

trainer.train(resume_from_checkpoint=resume)


Resume: /content/drive/MyDrive/ciftci/qwen3-sft-bf16/checkpoint-6891

Training başladı — loss değerleri akmalı (~10 sn/step)



Step,Training Loss,Validation Loss


TrainOutput(global_step=6891, training_loss=0.0, metrics={'train_runtime': 0.0085, 'train_samples_per_second': 6505911.294, 'train_steps_per_second': 813548.818, 'total_flos': 3.960935919022326e+17, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
# === 7. Final kaydet ===
final_path = f'{OUTPUT_DIR}/final'
trainer.save_model(final_path)
tok.save_pretrained(final_path)
print(f'\n✓ Eğitim bitti. Model: {final_path}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Eğitim bitti. Model: /content/drive/MyDrive/ciftci/qwen3-sft-bf16/final


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Model inference moduna alınıyor...")

# Modeli egitim modundan cikarip uretim moduna aliyoruz
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

test_tok = AutoTokenizer.from_pretrained(final_path)

# Eger ChatML ozel tokenleri varsa uretimi durdurmasi icin bunlari ekleyelim
stop_token_ids = [test_tok.eos_token_id]
if "<|im_end|>" in test_tok.vocab:
    stop_token_ids.append(test_tok.vocab["<|im_end|>"])

def ask_model(question, system_prompt="Sen Türk tarımı, hayvancılığı, tarımsal mevzuat ve çiftçi desteklerinde uzman bir asistansın. Çiftçilere pratik, doğru ve Türkçe cevaplar verirsin."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    # ChatML formatina cevir
    prompt = test_tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = test_tok(prompt, return_tensors="pt").to(model.device)

    # Yanit uret (generation)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3, # Daha kararli cevaplar icin dusurduk
            top_p=0.9,
            repetition_penalty=1.2, # Tekrar cezasini biraz daha artirdik
            do_sample=True,
            pad_token_id=test_tok.pad_token_id,
            eos_token_id=stop_token_ids
        )

    # Girdiyi kesip sadece uretilen cevabi dondur
    input_length = inputs.input_ids.shape[1]
    response = test_tok.decode(outputs[0][input_length:], skip_special_tokens=True)
    return response

# --- Test edelim ---
soru = "Buğday verimini artırmak için hangi gübreleri kullanmalıyım?"
print("\n" + "-"*50)
print(f"SORU: {soru}")
print("-"*50)
cevap = ask_model(soru)
print(f"CEVAP:\n{cevap}")
print("-"*50)


Model inference moduna alınıyor...

--------------------------------------------------
SORU: Buğday verimini artırmak için hangi gübreleri kullanmalıyım?
--------------------------------------------------
CEVAP:
<think>

</think>

Araştırmalar, kimyasal gübre uygulamalarının buğday çeşitlerinin tane verimi üzerindeki etkilerini incelemişti. Hangi gübrenin en iyi sonucu vereceği, kullanılan miktarına bağlıdır.
--------------------------------------------------


In [ ]:
print("\n" + "="*50)
print("🌾 Tarımsal Asistan Sohbet Robotu (Hafızalı)")
print("Çıkmak için 'q', 'çıkış' veya 'exit' yazın.")
print("="*50)

# Sohbet geçmişini başlatıyoruz
system_prompt = "Sen Türk tarımı, hayvancılığı, tarımsal mevzuat ve çiftçi desteklerinde uzman bir asistansın. Çiftçilere pratik, doğru ve Türkçe cevaplar verirsin."
chat_history = [
    {"role": "system", "content": system_prompt}
]

while True:
    kullanici_sorusu = input("\nSiz: ")

    # Çıkış kontrolü
    if kullanici_sorusu.lower().strip() in ['q', 'çıkış', 'cikis', 'exit']:
        print("\nAsistan: Görüşmek üzere, bereketli hasatlar dilerim! 👋")
        break

    if not kullanici_sorusu.strip():
        continue

    # 1. Kullanıcının sorusunu geçmişe ekle
    chat_history.append({"role": "user", "content": kullanici_sorusu})

    # 2. Tüm geçmişi modele uygun formata çevir
    prompt = test_tok.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = test_tok(prompt, return_tensors="pt").to(model.device)

    print("Asistan düşünüyor...")

    # 3. Cevap üret
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512, # Sohbet için biraz daha uzun olabilir
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=test_tok.pad_token_id,
            eos_token_id=stop_token_ids
        )

    # 4. Sadece yeni üretilen kısmı al
    input_length = inputs.input_ids.shape[1]
    response = test_tok.decode(outputs[0][input_length:], skip_special_tokens=True)

    # 5. Modelin cevabını geçmişe ekle (Hafıza için kritik adım!)
    chat_history.append({"role": "assistant", "content": response.strip()})

    print(f"\nAsistan: {response.strip()}")
    print("-" * 50)



🌾 Tarımsal Asistan Sohbet Robotu (Hafızalı)
Çıkmak için 'q', 'çıkış' veya 'exit' yazın.

Siz: organik gübre mi kimyasal gübre mi kullanmalıyım?
Asistan düşünüyor...

Asistan: <think>

</think>

Metin sadece organik ve kimyasal gübrelere dair tablolar sunmaktadır; hangi gübrenin sizin için daha uygun olduğunuz kararınıza bağlıdır. Bu konuda Tarım ve Orman Bakanlığı'nın güncel rehberleri veya bölge bazlı uygulamalarına bakmanız önerilir.
--------------------------------------------------

Siz: yüksek verim için hangisi?
Asistan düşünüyor...

Asistan: <think>

</think>

Tablolara göre farklı bitkilerdeki sonuçlar değişmektedir; örneğin patateste 30 kg N/dekar ile en yüksek değer 614 olarak görülmüştür. Hangi ürününize ne kadar azot ihtiyacı olduğunu kontrol etmelisiniz.
--------------------------------------------------

Siz: havuç ekiyorum
Asistan düşünüyor...

Asistan: <think>

</think>

Havuç için kimyasal gübre (CM-2'den CM-5'e) ve organik gübreler arasında performans farkları bulunm